# SEC Browser in Colab

This notebook mirrors the desktop SEC browser as closely as possible inside Google Colab.

Workflow:
1. Upload one or more `.asc` or `.xls` SEC exports.
2. Filter and select traces in the **Browser** tab.
3. Adjust processing in **Plot** and styling in **Appearance**.
4. Preview the figure, then export SVG + PNG, Plotly HTML, XMGrace, or a reusable session JSON from **Export**.

Tip: `Export SVG + PNG` also writes a matching `.session.json` file so the same figure state can be restored later.


In [ ]:
!pip -q install matplotlib pandas plotly xlrd ipywidgets openpyxl

from collections import Counter
from datetime import datetime
from pathlib import Path
import json
import subprocess
import sys
import zipfile

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
from IPython.display import FileLink, clear_output, display
from google.colab import files

REPO_URL = "https://github.com/elektra773/sec-browser.git"
REPO_DIR = Path("/content/sec-browser")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

UPLOAD_DIR = REPO_DIR / "colab_uploads"
OUTPUT_DIR = REPO_DIR / "colab_outputs"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from plot_sec_curves import (
    DEFAULT_FIGSIZE,
    DEFAULT_LINEWIDTH,
    EXPORT_RASTER_DPI,
    PREVIEW_DPI,
    clean_label,
    export_plotly_html,
    export_xmgrace,
    extract_column_name,
    extract_run_date,
    find_top_peaks,
    process_trace,
    read_sec_file,
    render_sec_plot,
)

COLOR_OPTIONS = {
    "Navy": "#16324f",
    "Blue": "#1f77b4",
    "Sky": "#4ea5d9",
    "Teal": "#1b998b",
    "Cyan": "#17becf",
    "Turquoise": "#2ec4b6",
    "Mint": "#52b788",
    "Seafoam": "#74c69d",
    "Green": "#2ca02c",
    "Forest": "#1b5e20",
    "Pine": "#2d6a4f",
    "Olive": "#7a8b24",
    "Lime": "#8ac926",
    "Chartreuse": "#a7c957",
    "Gold": "#d4a017",
    "Mustard": "#e09f3e",
    "Sand": "#e9c46a",
    "Orange": "#ff7f0e",
    "Peach": "#f7b267",
    "Rust": "#c7511f",
    "Red": "#d62728",
    "Crimson": "#b22222",
    "Coral": "#e76f51",
    "Salmon": "#f28482",
    "Berry": "#b23a48",
    "Magenta": "#e83f6f",
    "Rose": "#ff5d8f",
    "Fuchsia": "#d81b60",
    "Hot Pink": "#d94f98",
    "Cerise": "#d65db1",
    "Mulberry": "#b565c2",
    "Orchid": "#c77dff",
    "Purple": "#7a5195",
    "Violet": "#9d4edd",
    "Lavender": "#b8a1ff",
    "Plum": "#8e5ea2",
    "Brown": "#8c564b",
    "Tan": "#b08968",
    "Mocha": "#6f4e37",
    "Black": "#222222",
}

FILTER_TOKEN_STOPWORDS = {
    "run",
    "rerun",
    "sample",
    "trace",
    "standard",
}

DEFAULT_COLOR_SEQUENCE = list(COLOR_OPTIONS.keys())


def list_sec_files(directory: Path) -> list[Path]:
    return sorted(
        [path for path in directory.iterdir() if path.suffix.lower() in {".asc", ".xls"}],
        key=lambda path: path.name.lower(),
    )


def normalize_output_stem(value: str) -> str:
    cleaned = (value or "").strip() or "sec_overlay"
    path = Path(cleaned)
    if path.suffix.lower() in {".svg", ".png", ".html", ".agr", ".json"}:
        path = path.with_suffix("")
    return str(path)


def parse_optional_float(value) -> float | None:
    value = str(value).strip()
    if not value:
        return None
    return float(value)


def parse_limits(min_value, max_value):
    min_text = str(min_value).strip()
    max_text = str(max_value).strip()
    if not min_text and not max_text:
        return None
    return (float(min_text), float(max_text))


def label_tokens(label: str) -> set[str]:
    tokens = set()
    for raw_token in label.lower().replace("/", " ").split():
        token = "".join(ch for ch in raw_token if ch.isalnum())
        if len(token) < 2 or token.isdigit() or token in FILTER_TOKEN_STOPWORDS:
            continue
        tokens.add(token)
    return tokens


def build_inventory(paths: list[Path]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "filename": path.name,
                "sample": clean_label(path),
                "column": extract_column_name(path),
                "date": extract_run_date(path),
                "type": path.suffix.lower().lstrip("."),
            }
            for path in paths
        ]
    )


def title_from_selection(paths: list[Path]) -> str:
    if not paths:
        return ""
    labels = sorted({clean_label(path) for path in paths})
    columns = sorted({extract_column_name(path) for path in paths if extract_column_name(path)})
    dates = sorted({extract_run_date(path) for path in paths if extract_run_date(path)})
    title = " vs ".join(labels[:2]) if len(labels) == 2 else ", ".join(labels[:3])
    if len(labels) > 3:
        title = f"{title} +{len(labels) - 3} more"
    if columns:
        title = f"{title} [{', '.join(columns)}]"
    if dates:
        title = f"{title} ({', '.join(dates)})"
    return title


print(f"Repo: {REPO_DIR}")
print(f"Uploads: {UPLOAD_DIR}")
print(f"Outputs: {OUTPUT_DIR}")


In [ ]:
state = {
    "inventory": pd.DataFrame(),
    "filtered_paths": [],
    "color_widgets": {},
    "peak_widgets": {},
    "generated_files": [],
    "suspend_refresh": False,
}

STYLE = {"description_width": "140px"}
FULL = widgets.Layout(width="100%")
HALF = widgets.Layout(width="48%")
THIRD = widgets.Layout(width="32%")
BUTTON_ROW = widgets.Layout(width="auto")


def with_refresh_suspended(fn, *args, **kwargs):
    state["suspend_refresh"] = True
    try:
        return fn(*args, **kwargs)
    finally:
        state["suspend_refresh"] = False


def should_skip_refresh() -> bool:
    return bool(state.get("suspend_refresh"))


upload_widget = widgets.FileUpload(
    accept=".asc,.xls",
    multiple=True,
    description="Upload SEC Files",
    layout=widgets.Layout(width="220px"),
)
session_upload_widget = widgets.FileUpload(
    accept=".json",
    multiple=False,
    description="Upload Session",
    layout=widgets.Layout(width="220px"),
)
refresh_button = widgets.Button(description="Refresh", button_style="info", layout=BUTTON_ROW)
select_visible_button = widgets.Button(description="Select Visible", layout=BUTTON_ROW)
clear_selection_button = widgets.Button(description="Clear", layout=BUTTON_ROW)
load_session_button = widgets.Button(description="Load Session", layout=BUTTON_ROW)
title_from_selection_button = widgets.Button(description="Title From Selection", layout=BUTTON_ROW)
save_session_button = widgets.Button(description="Save Session JSON", layout=BUTTON_ROW)
download_bundle_button = widgets.Button(description="Download Last Export", layout=BUTTON_ROW, disabled=True)

search_widget = widgets.Text(description="Search", layout=FULL, style=STYLE)
format_widget = widgets.Dropdown(options=["asc", "xls", "both"], value="asc", description="Load", style=STYLE)
quick_filter_widget = widgets.SelectMultiple(
    options=[],
    description="Quick filters",
    rows=6,
    layout=FULL,
    style={"description_width": "initial"},
)
file_select_widget = widgets.SelectMultiple(
    options=[],
    description="Traces",
    rows=16,
    layout=widgets.Layout(width="100%", height="320px"),
    style={"description_width": "initial"},
)

inventory_output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px"))
preview_output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px"))
message_output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px"))
generated_box = widgets.VBox([widgets.HTML("<i>No exports yet.</i>")])
color_box = widgets.VBox([widgets.HTML("<i>Select traces to customize colors.</i>")])
peak_box = widgets.VBox([widgets.HTML("<i>Select traces to manage peak labels.</i>")])


display_mode_widget = widgets.ToggleButtons(
    options=[("Normalized", "normalized"), ("Actual", "actual")],
    value="normalized",
    description="Absorbance",
    style=STYLE,
)
baseline_widget = widgets.Checkbox(value=True, description="Baseline subtract")
smooth_widget = widgets.BoundedIntText(value=5, min=1, max=101, description="Smooth", style=STYLE)
anchor_widget = widgets.Dropdown(options=["left-limit", "zero"], value="left-limit", description="Anchor", style=STYLE)
title_widget = widgets.Text(value="", description="Title", layout=FULL, style=STYLE)
xmin_widget = widgets.Text(value="", description="X min", layout=HALF, style=STYLE)
xmax_widget = widgets.Text(value="", description="X max", layout=HALF, style=STYLE)
ymin_widget = widgets.Text(value="", description="Y min", layout=HALF, style=STYLE)
ymax_widget = widgets.Text(value="", description="Y max", layout=HALF, style=STYLE)

fig_width_widget = widgets.FloatText(value=DEFAULT_FIGSIZE[0], description="Width", style=STYLE)
fig_height_widget = widgets.FloatText(value=DEFAULT_FIGSIZE[1], description="Height", style=STYLE)
line_width_widget = widgets.FloatText(value=DEFAULT_LINEWIDTH, description="Line width", style=STYLE)
style_widget = widgets.Dropdown(options=["paper", "talk"], value="paper", description="Style", style=STYLE)
show_legend_widget = widgets.Checkbox(value=True, description="Show legend")
transparent_widget = widgets.Checkbox(value=False, description="Transparent background")

output_stem_widget = widgets.Text(value="sec_overlay", description="Output stem", layout=FULL, style=STYLE)
preview_button = widgets.Button(description="Preview Matplotlib", button_style="success")
plotly_preview_button = widgets.Button(description="Preview Plotly")
export_svg_png_button = widgets.Button(description="Export SVG + PNG", button_style="primary")
export_plotly_button = widgets.Button(description="Export Plotly HTML")
export_xmgrace_button = widgets.Button(description="Export XMGrace")



def uploaded_items(widget):
    value = widget.value
    if not value:
        return []
    if isinstance(value, dict):
        return list(value.values())
    return list(value)



def save_uploaded_files(_change=None):
    items = uploaded_items(upload_widget)
    if not items:
        return
    saved = 0
    for item in items:
        name = item["name"]
        content = item["content"]
        if not isinstance(content, (bytes, bytearray)):
            content = content.tobytes()
        (UPLOAD_DIR / name).write_bytes(content)
        saved += 1
    with message_output:
        clear_output(wait=True)
        print(f"Saved {saved} SEC file(s) to {UPLOAD_DIR}")
    upload_widget.value.clear()
    upload_widget._counter = 0
    refresh_dataset()



def build_session_state() -> dict:
    selected_paths = [UPLOAD_DIR / name for name in file_select_widget.value]
    xlim = parse_limits(xmin_widget.value, xmax_widget.value)
    ylim = parse_limits(ymin_widget.value, ymax_widget.value)
    return {
        "saved_at": datetime.now().isoformat(timespec="seconds"),
        "output_base": str((OUTPUT_DIR / normalize_output_stem(output_stem_widget.value)).resolve()),
        "selected_traces": [str(path) for path in selected_paths],
        "trace_labels": {str(path): clean_label(path) for path in selected_paths},
        "trace_colors": {
            str(UPLOAD_DIR / name): COLOR_OPTIONS[state["color_widgets"][name].value]
            for name in file_select_widget.value
            if name in state["color_widgets"]
        },
        "display_mode": display_mode_widget.value,
        "normalize_anchor": anchor_widget.value,
        "baseline_subtract": baseline_widget.value,
        "smooth_window": int(smooth_widget.value),
        "style": style_widget.value,
        "line_width": float(line_width_widget.value),
        "figure_width": float(fig_width_widget.value),
        "figure_height": float(fig_height_widget.value),
        "show_legend": show_legend_widget.value,
        "title": title_widget.value,
        "xlim": list(xlim) if xlim else None,
        "ylim": list(ylim) if ylim else None,
        "transparent_background": transparent_widget.value,
        "load_format": format_widget.value,
        "active_filters": list(quick_filter_widget.value),
        "peak_visibility": {
            str(UPLOAD_DIR / name): sorted(int(rank) for rank in state["peak_widgets"][name].value)
            for name in file_select_widget.value
            if name in state["peak_widgets"] and state["peak_widgets"][name].value
        },
    }



def apply_session_state(session_state: dict):
    selected_names = []
    for path_text in session_state.get("selected_traces", []):
        selected_names.append(Path(path_text).name)

    trace_colors = {
        Path(path_text).name: color
        for path_text, color in session_state.get("trace_colors", {}).items()
    }
    peak_visibility = {
        Path(path_text).name: tuple(int(rank) for rank in ranks)
        for path_text, ranks in session_state.get("peak_visibility", {}).items()
    }

    def assign_widget_values():
        output_base = session_state.get("output_base", "")
        if output_base:
            try:
                relative = Path(output_base).relative_to(OUTPUT_DIR)
                output_stem_widget.value = normalize_output_stem(str(relative))
            except ValueError:
                output_stem_widget.value = normalize_output_stem(Path(output_base).name)
        display_mode_widget.value = session_state.get("display_mode", display_mode_widget.value)
        anchor_widget.value = session_state.get("normalize_anchor", anchor_widget.value)
        baseline_widget.value = session_state.get("baseline_subtract", baseline_widget.value)
        smooth_widget.value = int(session_state.get("smooth_window", smooth_widget.value))
        style_widget.value = session_state.get("style", style_widget.value)
        line_width_widget.value = float(session_state.get("line_width", line_width_widget.value))
        fig_width_widget.value = float(session_state.get("figure_width", fig_width_widget.value))
        fig_height_widget.value = float(session_state.get("figure_height", fig_height_widget.value))
        show_legend_widget.value = bool(session_state.get("show_legend", show_legend_widget.value))
        title_widget.value = session_state.get("title", title_widget.value)
        transparent_widget.value = bool(session_state.get("transparent_background", transparent_widget.value))
        format_widget.value = session_state.get("load_format", format_widget.value)
        xlim = session_state.get("xlim") or ["", ""]
        ylim = session_state.get("ylim") or ["", ""]
        xmin_widget.value = "" if xlim[0] in (None, "") else str(xlim[0])
        xmax_widget.value = "" if xlim[1] in (None, "") else str(xlim[1])
        ymin_widget.value = "" if ylim[0] in (None, "") else str(ylim[0])
        ymax_widget.value = "" if ylim[1] in (None, "") else str(ylim[1])

    with_refresh_suspended(assign_widget_values)
    refresh_dataset()

    available_filters = set(quick_filter_widget.options)
    desired_filters = tuple(token for token in session_state.get("active_filters", []) if token in available_filters)
    with_refresh_suspended(lambda: setattr(quick_filter_widget, "value", desired_filters))
    refresh_dataset()

    available_names = {value for _label, value in file_select_widget.options}
    restored_names = tuple(name for name in selected_names if name in available_names)
    with_refresh_suspended(lambda: setattr(file_select_widget, "value", restored_names))
    rebuild_dynamic_controls()

    for name, color in trace_colors.items():
        widget = state["color_widgets"].get(name)
        if widget is None:
            continue
        for color_name, color_hex in COLOR_OPTIONS.items():
            if color_hex.lower() == str(color).lower():
                widget.value = color_name
                break

    for name, ranks in peak_visibility.items():
        widget = state["peak_widgets"].get(name)
        if widget is not None:
            widget.value = tuple(rank for rank in ranks if rank in {1, 2, 3, 4, 5})

    with message_output:
        clear_output(wait=True)
        print(f"Loaded session with {len(restored_names)} restored trace(s).")



def load_uploaded_session(_=None):
    items = uploaded_items(session_upload_widget)
    if not items:
        with message_output:
            clear_output(wait=True)
            print("Upload a .json session file first.")
        return
    item = items[0]
    content = item["content"]
    if not isinstance(content, (bytes, bytearray)):
        content = content.tobytes()
    try:
        session_state = json.loads(content.decode("utf-8"))
    except Exception as exc:
        with message_output:
            clear_output(wait=True)
            print(f"Could not read session JSON: {exc}")
        return
    apply_session_state(session_state)
    session_upload_widget.value.clear()
    session_upload_widget._counter = 0



def save_session_json(_=None):
    output_base = OUTPUT_DIR / normalize_output_stem(output_stem_widget.value)
    output_base.parent.mkdir(parents=True, exist_ok=True)
    session_output = output_base.with_suffix(".session.json")
    session_output.write_text(json.dumps(build_session_state(), indent=2))
    update_generated_files([session_output])
    with message_output:
        clear_output(wait=True)
        print(f"Saved session JSON to {session_output}")



def refresh_dataset(*_args):
    if should_skip_refresh():
        return

    sec_files = list_sec_files(UPLOAD_DIR)
    inventory = build_inventory(sec_files)
    state["inventory"] = inventory

    token_counts = Counter()
    for path in sec_files:
        token_counts.update(label_tokens(clean_label(path)))
    current_filters = set(quick_filter_widget.value)
    quick_tokens = [token for token, count in token_counts.most_common(12) if count >= 2]

    def update_filter_widgets():
        quick_filter_widget.options = quick_tokens
        quick_filter_widget.value = tuple(token for token in quick_tokens if token in current_filters)

    with_refresh_suspended(update_filter_widgets)

    query = search_widget.value.strip().lower()
    load_type = format_widget.value
    selected_filters = set(quick_filter_widget.value)
    filtered_paths = []
    for path in sec_files:
        label = clean_label(path)
        if query and query not in label.lower() and query not in path.name.lower():
            continue
        if load_type != "both" and path.suffix.lower().lstrip(".") != load_type:
            continue
        if selected_filters and not (label_tokens(label) & selected_filters):
            continue
        filtered_paths.append(path)
    state["filtered_paths"] = filtered_paths

    previously_selected = set(file_select_widget.value)
    file_options = [
        (
            f"{clean_label(path)} [{extract_column_name(path) or 'No column'} | {extract_run_date(path) or 'No date'} | {path.suffix.lower().lstrip('.')} ]",
            path.name,
        )
        for path in filtered_paths
    ]
    visible_names = {path.name for path in filtered_paths}

    def update_file_selection():
        file_select_widget.options = file_options
        file_select_widget.value = tuple(name for name in previously_selected if name in visible_names)

    with_refresh_suspended(update_file_selection)
    rebuild_dynamic_controls()

    with inventory_output:
        clear_output(wait=True)
        if inventory.empty:
            print("No SEC files uploaded yet.")
        else:
            filtered_names = {path.name for path in filtered_paths}
            filtered_inventory = inventory[inventory["filename"].isin(filtered_names)]
            print(f"Showing {len(filtered_inventory)} of {len(inventory)} uploaded SEC file(s).")
            display(filtered_inventory.reset_index(drop=True))



def rebuild_dynamic_controls(*_args):
    selected_names = list(file_select_widget.value)
    selected_paths = [UPLOAD_DIR / name for name in selected_names]

    color_rows = []
    for idx, path in enumerate(selected_paths):
        widget = state["color_widgets"].get(path.name)
        if widget is None:
            widget = widgets.Dropdown(
                options=list(COLOR_OPTIONS.keys()),
                value=DEFAULT_COLOR_SEQUENCE[idx % len(DEFAULT_COLOR_SEQUENCE)],
                description="",
                layout=widgets.Layout(width="220px"),
            )
            state["color_widgets"][path.name] = widget
        color_rows.append(
            widgets.HBox([
                widgets.HTML(f"<div style='min-width:320px'>{clean_label(path)}</div>"),
                widget,
            ])
        )
    color_box.children = tuple(color_rows) if color_rows else (widgets.HTML("<i>Select traces to customize colors.</i>"),)

    peak_rows = []
    for path in selected_paths:
        widget = state["peak_widgets"].get(path.name)
        if widget is None:
            widget = widgets.SelectMultiple(
                options=[(f"Peak {rank}", rank) for rank in range(1, 6)],
                value=(),
                rows=5,
                layout=widgets.Layout(width="220px", height="120px"),
            )
            state["peak_widgets"][path.name] = widget
        peak_rows.append(
            widgets.HBox([
                widgets.HTML(f"<div style='min-width:320px'>{clean_label(path)}</div>"),
                widget,
            ])
        )
    peak_box.children = tuple(peak_rows) if peak_rows else (widgets.HTML("<i>Select traces to manage peak labels.</i>"),)



def build_processed_traces():
    selected_names = list(file_select_widget.value)
    if not selected_names:
        raise ValueError("Select at least one trace.")

    xlim = parse_limits(xmin_widget.value, xmax_widget.value)
    ylim = parse_limits(ymin_widget.value, ymax_widget.value)
    processed_traces = []
    peak_visibility = {}

    for name in selected_names:
        path = UPLOAD_DIR / name
        color_widget = state["color_widgets"].get(name)
        peak_widget = state["peak_widgets"].get(name)
        color_name = color_widget.value if color_widget is not None else DEFAULT_COLOR_SEQUENCE[0]
        color_hex = COLOR_OPTIONS[color_name]
        peak_visibility[str(path)] = set(peak_widget.value) if peak_widget is not None else set()
        raw_trace = read_sec_file(path)
        trace = process_trace(
            raw_trace,
            normalize=display_mode_widget.value == "normalized",
            baseline_subtract=baseline_widget.value,
            smooth_window=max(1, int(smooth_widget.value)),
            normalize_window=xlim if display_mode_widget.value == "normalized" else None,
            normalize_anchor=anchor_widget.value,
        )
        processed_traces.append(
            {
                "trace_id": str(path),
                "label": clean_label(path),
                "color": color_hex,
                "line_width": max(0.4, float(line_width_widget.value)),
                "data": trace,
                "peaks": find_top_peaks(trace, peak_window=xlim, top_n=5),
            }
        )
    return processed_traces, xlim, ylim, peak_visibility



def render_current_figure(figsize=None):
    processed_traces, xlim, ylim, peak_visibility = build_processed_traces()
    figsize = figsize or (float(fig_width_widget.value), float(fig_height_widget.value))
    fig, ax = plt.subplots(figsize=figsize, dpi=PREVIEW_DPI)
    render_sec_plot(
        fig=fig,
        ax=ax,
        traces=processed_traces,
        title=title_widget.value.strip(),
        normalized=display_mode_widget.value == "normalized",
        xlim=xlim,
        ylim=ylim,
        show_legend=show_legend_widget.value,
        style=style_widget.value,
        visible_peak_ranks=peak_visibility,
    )
    return fig, processed_traces, xlim, ylim



def preview_matplotlib(_=None):
    with preview_output:
        clear_output(wait=True)
        try:
            fig, _processed_traces, _xlim, _ylim = render_current_figure()
        except Exception as exc:
            print(exc)
            return
        plt.show()



def preview_plotly(_=None):
    with preview_output:
        clear_output(wait=True)
        try:
            processed_traces, xlim, ylim, _peak_visibility = build_processed_traces()
        except Exception as exc:
            print(exc)
            return
        figure = go.Figure()
        for trace in processed_traces:
            figure.add_trace(
                go.Scatter(
                    x=trace["data"]["ml"],
                    y=trace["data"]["mAU"],
                    mode="lines",
                    name=trace["label"],
                    line={"color": trace["color"], "width": trace["line_width"]},
                )
            )
        figure.update_layout(
            title=title_widget.value.strip(),
            xaxis_title="Elution Volume (mL)",
            yaxis_title="Normalized absorbance" if display_mode_widget.value == "normalized" else "Absorbance (mAU)",
            showlegend=show_legend_widget.value,
            template="simple_white",
            font={"family": "Arial, Helvetica, sans-serif", "size": 14},
        )
        if xlim:
            figure.update_xaxes(range=list(xlim))
        if ylim:
            figure.update_yaxes(range=list(ylim))
        figure.show()



def update_generated_files(paths):
    state["generated_files"] = list(paths)
    generated_children = []
    for path in paths:
        generated_children.append(widgets.HTML(FileLink(str(path))._repr_html_()))
    generated_box.children = tuple(generated_children) if generated_children else (widgets.HTML("<i>No exports yet.</i>"),)
    download_bundle_button.disabled = not bool(paths)



def export_static(include_svg_png=False, include_plotly=False, include_xmgrace=False):
    generated = []
    with message_output:
        clear_output(wait=True)
        try:
            fig, processed_traces, xlim, ylim = render_current_figure()
        except Exception as exc:
            print(exc)
            return

        output_base = OUTPUT_DIR / normalize_output_stem(output_stem_widget.value)
        output_base.parent.mkdir(parents=True, exist_ok=True)

        if include_svg_png:
            svg_output = output_base.with_suffix(".svg")
            png_output = output_base.with_suffix(".png")
            session_output = output_base.with_suffix(".session.json")
            fig.savefig(svg_output, transparent=transparent_widget.value)
            fig.savefig(png_output, dpi=EXPORT_RASTER_DPI, transparent=transparent_widget.value)
            session_output.write_text(json.dumps(build_session_state(), indent=2))
            generated.extend([svg_output, png_output, session_output])

        plt.close(fig)

        if include_plotly:
            html_output = output_base.with_suffix(".html")
            export_plotly_html(
                traces=processed_traces,
                output=html_output,
                title=title_widget.value.strip(),
                x_label="Elution Volume (mL)",
                y_label="Normalized absorbance" if display_mode_widget.value == "normalized" else "Absorbance (mAU)",
                show_legend=show_legend_widget.value and len(processed_traces) > 1,
                xlim=xlim,
                ylim=ylim,
            )
            generated.append(html_output)

        if include_xmgrace:
            agr_output = output_base.with_suffix(".agr")
            export_xmgrace(
                traces=processed_traces,
                output=agr_output,
                title=title_widget.value.strip(),
                x_label="Elution Volume (mL)",
                y_label="Normalized absorbance" if display_mode_widget.value == "normalized" else "Absorbance (mAU)",
                show_legend=show_legend_widget.value and len(processed_traces) > 1,
                xlim=xlim,
                ylim=ylim,
            )
            generated.append(agr_output)

        print("Generated files:")
        for path in generated:
            print(path)

    update_generated_files(generated)



def export_svg_png(_=None):
    export_static(include_svg_png=True)



def export_plotly_file(_=None):
    export_static(include_plotly=True)



def export_xmgrace_file(_=None):
    export_static(include_xmgrace=True)



def download_last_bundle(_=None):
    generated = state.get("generated_files", [])
    if not generated:
        with message_output:
            clear_output(wait=True)
            print("Nothing exported yet.")
        return
    bundle_path = OUTPUT_DIR / "sec_browser_export_bundle.zip"
    with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in generated:
            bundle.write(path, arcname=Path(path).name)
    with message_output:
        clear_output(wait=True)
        print(f"Downloading {bundle_path}")
    files.download(str(bundle_path))



def title_from_current_selection(_=None):
    paths = [UPLOAD_DIR / name for name in file_select_widget.value]
    title_widget.value = title_from_selection(paths)



def select_visible(_=None):
    file_select_widget.value = tuple(value for _label, value in file_select_widget.options)



def clear_selection(_=None):
    file_select_widget.value = ()



def maybe_refresh(_change=None):
    if not should_skip_refresh():
        refresh_dataset()


upload_widget.observe(save_uploaded_files, names="value")
refresh_button.on_click(refresh_dataset)
select_visible_button.on_click(select_visible)
clear_selection_button.on_click(clear_selection)
load_session_button.on_click(load_uploaded_session)
title_from_selection_button.on_click(title_from_current_selection)
save_session_button.on_click(save_session_json)
download_bundle_button.on_click(download_last_bundle)
preview_button.on_click(preview_matplotlib)
plotly_preview_button.on_click(preview_plotly)
export_svg_png_button.on_click(export_svg_png)
export_plotly_button.on_click(export_plotly_file)
export_xmgrace_button.on_click(export_xmgrace_file)
search_widget.observe(maybe_refresh, names="value")
format_widget.observe(maybe_refresh, names="value")
quick_filter_widget.observe(maybe_refresh, names="value")
file_select_widget.observe(rebuild_dynamic_controls, names="value")

browser_tab = widgets.VBox([
    widgets.HTML("<b>Upload</b>"),
    widgets.HBox([upload_widget, session_upload_widget]),
    widgets.HBox([refresh_button, select_visible_button, clear_selection_button, load_session_button]),
    search_widget,
    format_widget,
    widgets.HTML("<b>Quick Filters</b>"),
    quick_filter_widget,
    widgets.HTML("<b>Trace Selection</b>"),
    file_select_widget,
])

plot_tab = widgets.VBox([
    widgets.HTML("<b>Trace Processing</b>"),
    display_mode_widget,
    widgets.HBox([baseline_widget, smooth_widget, anchor_widget]),
    title_widget,
    widgets.HBox([title_from_selection_button]),
    widgets.HTML("<b>Axis Limits</b>"),
    widgets.HBox([xmin_widget, xmax_widget]),
    widgets.HBox([ymin_widget, ymax_widget]),
    widgets.HTML("<b>Peak Annotations</b><br><span style='font-size:12px;color:#666'>Top 5 peaks are found inside the current X limits when limits are set.</span>"),
    peak_box,
])

appearance_tab = widgets.VBox([
    widgets.HTML("<b>Figure Styling</b>"),
    widgets.HBox([fig_width_widget, fig_height_widget, line_width_widget]),
    widgets.HBox([style_widget, show_legend_widget, transparent_widget]),
    widgets.HTML("<b>Trace Colors</b>"),
    color_box,
])

export_tab = widgets.VBox([
    widgets.HTML("<b>Preview and Export</b>"),
    output_stem_widget,
    widgets.HBox([preview_button, plotly_preview_button]),
    widgets.HBox([export_svg_png_button, export_plotly_button, export_xmgrace_button]),
    widgets.HBox([save_session_button, download_bundle_button]),
    widgets.HTML("<b>Generated Files</b>"),
    generated_box,
])

main_tabs = widgets.Tab(children=[browser_tab, plot_tab, appearance_tab, export_tab])
main_tabs.set_title(0, "Browser")
main_tabs.set_title(1, "Plot")
main_tabs.set_title(2, "Appearance")
main_tabs.set_title(3, "Export")

summary = widgets.HTML(
    "<div style='padding:8px 0 12px 0'><b>SEC Browser for Colab</b><br>Upload SEC files, filter them, preview the overlay, and export the same output formats as the desktop browser.</div>"
)

refresh_dataset()

display(summary)
display(main_tabs)
display(widgets.HTML("<h4>Inventory</h4>"))
display(inventory_output)
display(widgets.HTML("<h4>Preview</h4>"))
display(preview_output)
display(widgets.HTML("<h4>Status</h4>"))
display(message_output)
